<a href="https://colab.research.google.com/github/Alaxooy/Programming-for-data-analytics/blob/main/Fake_not_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [40]:
import pandas as pd
import numpy as np
columns = ['variance', 'skewness', 'curtosis', 'entropy', 'class']
df = pd.read_csv('/content/data_banknote_authentication[1].txt', names=columns)

In [41]:
df.head(5)

,variance,skewness,curtosis,entropy,class
0,3.62160,8.6661,-2.8073,-0.44699,0
1,4.54590,8.1674,-2.4586,-1.46210,0
2,3.86600,-2.6383,1.9242,0.10645,0
3,3.45660,9.5228,-4.0112,-3.59440,0
4,0.32924,-4.4552,4.5718,-0.98880,0


In [42]:
df.describe()

,variance,skewness,curtosis,entropy,class
count,1372.000000,1372.000000,1372.000000,1372.000000,1372.000000
mean,0.433735,1.922353,1.397627,-1.191657,0.444606
std,2.842763,5.869047,4.310030,2.101013,0.497103
min,-7.042100,-13.773100,-5.286100,-8.548200,0.000000
25%,-1.773000,-1.708200,-1.574975,-2.413450,0.000000
50%,0.496180,2.319650,0.616630,-0.586650,0.000000
75%,2.821475,6.814625,3.179250,0.394810,1.000000
max,6.824800,12.951600,17.927400,2.449500,1.000000


In [46]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

X = df[['variance', 'skewness', 'curtosis', 'entropy']]
y = df['class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
model = BernoulliNB()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

In [47]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}\n")

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Genuine (0)', 'Counterfeit (1)']))
false_negatives = cm[1, 0]
print(f"Number of counterfeit banknotes incorrectly classified as genuine (False Negatives): {false_negatives}")

Accuracy: 0.8618

Confusion Matrix:
[[133  20]
 [ 18 104]]

Classification Report:
                 precision    recall  f1-score   support

    Genuine (0)       0.88      0.87      0.88       153
Counterfeit (1)       0.84      0.85      0.85       122

       accuracy                           0.86       275
      macro avg       0.86      0.86      0.86       275
   weighted avg       0.86      0.86      0.86       275

Number of counterfeit banknotes incorrectly classified as genuine (False Negatives): 18


In [38]:
original_accuracy = accuracy
features = ['variance', 'skewness', 'curtosis', 'entropy']
feature_drop_results = {}

for feature in features:
    remaining_features = [f for f in features if f != feature]
    X_train_reduced = X_train[remaining_features]
    X_test_reduced = X_test[remaining_features]
    reduced_model = GaussianNB()
    reduced_model.fit(X_train_reduced, y_train)
    y_pred_reduced = reduced_model.predict(X_test_reduced)
    reduced_accuracy = accuracy_score(y_test, y_pred_reduced)
    accuracy_drop = original_accuracy - reduced_accuracy
    feature_drop_results[feature] = {
        'Accuracy': reduced_accuracy,
        'Drop in Accuracy': accuracy_drop
    }

print(f"Original Model Accuracy (All 4 features): {original_accuracy:.4f}\n")
print(f"{'Feature Removed':<20} | {'New Accuracy':<12} | {'Drop in Accuracy':<16}")
print("-" * 54)
for feat, stats in feature_drop_results.items():
    print(f"{feat:<20} | {stats['Accuracy']:<12.4f} | {stats['Drop in Accuracy']:<16.4f}")
worst_feature = max(feature_drop_results, key=lambda k: feature_drop_results[k]['Drop in Accuracy'])
print(f"\nThe feature that causes the greatest reduction in performance when removed is: **{worst_feature}**")

Original Model Accuracy (All 4 features): 0.8618

Feature Removed      | New Accuracy | Drop in Accuracy
------------------------------------------------------
variance             | 0.5927       | 0.2691          
skewness             | 0.8618       | 0.0000          
curtosis             | 0.8764       | -0.0145         
entropy              | 0.8582       | 0.0036          

The feature that causes the greatest reduction in performance when removed is: **variance**
